In [1]:
import tensorflow as tf
from tensorflow.python.ops.gradient_checker_v2 import max_error

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
  for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

import warnings
import os 
warnings.filterwarnings("ignore")
import config
from utils import *
import math 
from tensorflow.keras.callbacks import TerminateOnNaN, EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from numba import cuda 

from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
tf.keras.backend.clear_session()

os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"




2025-03-08 20:32:31.006432: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1741462351.027759   26708 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1741462351.031919   26708 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-08 20:32:31.045861: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


{'Cargo plane': 635, 'Helicopter': 70, 'Small car': 4290, 'Bus': 2155, 'Truck': 2746, 'Motorboat': 1069, 'Fishing vessel': 706, 'Dump truck': 1236, 'Excavator': 789, 'Building': 4689, 'Storage tank': 1469, 'Shipping container': 1523}


I0000 00:00:1741462354.158585   26708 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6324 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1070, pci bus id: 0000:02:00.0, compute capability: 6.1


In [2]:
import math
import gc
from tensorflow.keras.callbacks import TerminateOnNaN, EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from keras_tuner import BayesianOptimization
from tensorflow.keras.callbacks import ModelCheckpoint



# Run the hyperparameter search
batch_size = 256
train_generator, valid_generator, sz_train, sz_val = train_val_split(train_batch=batch_size)
train_steps = math.ceil(sz_train / batch_size)
valid_steps = math.ceil(sz_val / batch_size)

# Set up Bayesian Optimizer
tuner = BayesianOptimization(
    build_fcnn,
    objective='val_accuracy',
    max_trials=60,  # Number of total trials to run
    directory='bayesian_search',
    project_name='fcnn_tuning',
    overwrite=True,
    # max_model_size=1_000_000_000,
    max_consecutive_failed_trials=2,
    executions_per_trial=1  # Disallow parallel execution
)

# Define callback for the search
early_stop_tuner = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True
)
tuner.search(
    train_generator,
    steps_per_epoch=train_steps,
    validation_data=valid_generator,
    validation_steps=valid_steps,
    epochs=25,
    callbacks=[early_stop_tuner]
)

# Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

Trial 9 Complete [00h 00m 10s]

Best val_accuracy So Far: 0.29794201254844666
Total elapsed time: 00h 10m 29s

Search: Running Trial #10

Value             |Best Value So Far |Hyperparameter
3                 |3                 |num_layers
960               |896               |neurons_0
0.042905          |0.00011987        |l1_0
0.00057899        |0.00088371        |l2_0
leaky_relu        |elu               |activation_0
glorot            |glorot            |init_0
False             |False             |batch_norm_0
0                 |0.2               |dropout_0
960               |640               |neurons_1
0.0028016         |0.0012229         |l1_1
0.059839          |0.00021083        |l2_1
elu               |elu               |activation_1
glorot            |glorot            |init_1
True              |True              |batch_norm_1
0.1               |0.4               |dropout_1
768               |512               |neurons_2
0.005971          |0.0044642         |l1_2
0.00041531 

2025-03-08 20:43:14.597193: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:497] Allocator (GPU_0_bfc) ran out of memory trying to allocate 551.25MiB (rounded to 578027520)requested by op StatelessRandomUniformV2
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-03-08 20:43:14.597789: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1053] BFCAllocator dump for GPU_0_bfc
2025-03-08 20:43:14.597849: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1060] Bin (256): 	Total Chunks: 10615, Chunks in use: 10614. 2.59MiB allocated for chunks. 2.59MiB in use in bin. 42.5KiB client-requested in use in bin.
2025-03-08 20:43:14.597874: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1060] Bin (512): 	Total Chunks: 0, Chunks in use: 0. 0B allocated for chunks. 0B in use in bin. 0B client-requested in use in bin

RuntimeError: Number of consecutive failures exceeded the limit of 5.
Traceback (most recent call last):
  File "/home/fran/.virtualenvs/DeepLearning/lib/python3.12/site-packages/keras_tuner/src/engine/base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "/home/fran/.virtualenvs/DeepLearning/lib/python3.12/site-packages/keras_tuner/src/engine/base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/fran/.virtualenvs/DeepLearning/lib/python3.12/site-packages/keras_tuner/src/tuners/hyperband.py", line 427, in run_trial
    return super().run_trial(trial, *fit_args, **fit_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/fran/.virtualenvs/DeepLearning/lib/python3.12/site-packages/keras_tuner/src/engine/tuner.py", line 314, in run_trial
    obj_value = self._

In [ ]:
import math

layers = [
    LayerSettings(512, ('l2', 0.01), 'relu', 'he', 0.2, True),
    LayerSettings(512, ('l2', 0.01), 'relu', 'he', 0.2, True),
    LayerSettings(512, ('l2', 0.01), 'relu', 'he', 0.2, True)
]
model, name = generate_fcnn_model(layers)

from tensorflow.keras.callbacks import TerminateOnNaN, EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

model_checkpoint = ModelCheckpoint(f'{name}.keras', monitor='val_accuracy', verbose=1, save_best_only=True)
reduce_lr = ReduceLROnPlateau('val_accuracy', factor=0.1, patience=10, verbose=1)
early_stop = EarlyStopping('val_accuracy', patience=40, verbose=1)
terminate = TerminateOnNaN()
callbacks = [model_checkpoint, reduce_lr, early_stop, terminate]

callbacks = callbacks + [TimingCallback(), HistorySaverCallback()]

batch_size = 256
train_generator, valid_generator, sz_train, sz_val = train_val_split(batch_size=batch_size)
epochs = 80
train_steps = math.ceil(sz_train / batch_size)
valid_steps = math.ceil(sz_val / batch_size)
h = model.fit(train_generator,
              steps_per_epoch=train_steps,
              validation_data=valid_generator,
              validation_steps=valid_steps,
              epochs=epochs,
              callbacks=callbacks,
              verbose=1,
              )
# Best validation model
best_idx = int(np.argmax(h.history['val_accuracy']))
best_value = np.max(h.history['val_accuracy'])
print('Best validation model: epoch ' + str(best_idx + 1), ' - val_accuracy ' + str(best_value))